# Experiment 3: Class-Weighted Cross-Entropy
**GPU:** P100 | **Estimated time:** ~2 hours

In [ ]:
# Cell 1: Setup
!pip install -q pytorch-crf sentencepiece gdown
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('NO GPU DETECTED! Stop and switch to GPU runtime.')

In [ ]:
# Cell 2: Clone repo and prepare workspace
import os, shutil

REPO_URL = "https://github.com/Obyedullahilmamun/Punctuation-Restoration-Bangla.git"
BRANCH = "focal-loss-experiment"
REPO_DIR = "Punctuation-Restoration-Bangla"
WEIGHTS_ID = "1X2udyT1XYrmCNvWtFpT_6jrWsQejGCBW"

!rm -rf {REPO_DIR}
!git clone -b {BRANCH} {REPO_URL}

os.makedirs(f"{REPO_DIR}/weights", exist_ok=True)
os.makedirs(f"{REPO_DIR}/data/test", exist_ok=True)

bn_data_dir = f"{REPO_DIR}/data/bn"
for file in os.listdir(bn_data_dir):
    if file.startswith("test_"):
        shutil.copy(os.path.join(bn_data_dir, file), f"{REPO_DIR}/data/test/{file}")
        print(f"  Copied {file} -> data/test/")

!gdown {WEIGHTS_ID} -O {REPO_DIR}/weights/xlm-roberta-large-bn.pt
print("\n✅ Workspace ready!")

In [ ]:
# Cell 3: TRAIN — Class-Weighted CE
%cd /kaggle/working/{REPO_DIR}

!python src/train.py \
    --cuda=True \
    --pretrained-model=xlm-roberta-large \
    --freeze-bert=False \
    --lstm-dim=-1 \
    --language=bangla \
    --seed=1 \
    --lr=5e-6 \
    --epoch=3 \
    --use-crf=False \
    --augment-type=all \
    --augment-rate=0.20 \
    --alpha-sub=0.4 \
    --alpha-del=0.4 \
    --loss-type=weighted-ce \
    --data-path=data \
    --save-path=out-weighted-ce

In [ ]:
# Cell 4: EVALUATE
!python src/test.py \
    --pretrained-model=xlm-roberta-large \
    --lstm-dim=-1 \
    --use-crf=False \
    --data-path=data/test \
    --weight-path=out-weighted-ce/weights.pt \
    --sequence-length=256 \
    --save-path=out-weighted-ce

print("\n✅ Experiment 3 (Weighted-CE) complete!")
print("Results saved in out-weighted-ce/")

# Show results
!cat out-weighted-ce/logs.txt